In [1]:
import sys
sys.path.append("/global/u1/r/rmastand/mlpf-ssl/particlemind")
# !{sys.executable} -m pip install awkward

import os

# Force Hugging Face datasets cache to local scratch (avoid NFS filelock hangs)
os.environ["HF_DATASETS_CACHE"] = f"/tmp/{os.environ['USER']}/hf_datasets_cache"
os.environ["HF_HOME"] = f"/tmp/{os.environ['USER']}/hf_home"  # optional, safer for configs


In [2]:
import numpy as np
import matplotlib.pyplot as plt
import torch  # CHANGE: moved torch import here with other top-level imports

import awkward as ak
from src.data.CaloHitDataset import CaloHitDataset
from src.data.CaloPatchDataset import CaloPatchDataset
from src.data.augmentations import standardize_calo_hit_features_rphiz, augment_data

from src.models.vqvae import VQVAELightning
from src.data.utils import CollaterPatch, CollaterHits
from torch.utils.data import DataLoader
from tqdm import tqdm


In [3]:
num_files = 1

E_min = 0.001
pad = 1000

data_type = "hit"

if data_type == "hit":
    collater = CollaterHits(empty_key="calo_hit_features", variable_size_keys="all", pad=pad)
    loader = CaloHitDataset

elif data_type == "patch":
    collater = CollaterPatch  # CHANGE: renamed collator -> collater for consistency
    loader = CaloPatchDataset


In [4]:
paths_to_generated_calo = [
    f"/pscratch/sd/r/rmastand/particlemind/data/generated_colliderML_parquetfiles/generated_{i}.parquet"
    for i in range(num_files)
]

path_to_embedder_checkpoint = "/pscratch/sd/r/rmastand/particlemind/vqvae_training/best_models/embedder_hit_vqvae_val_loss_epoch=00-v3.ckpt"


# Apply embedder to original data

tokenize and reconstruct

In [5]:
# Load embedder from checkpoint
embedder = VQVAELightning.load_from_checkpoint(path_to_embedder_checkpoint)

# CHANGE: replaced 8 separate ak.Array([]) initializations with a single dict for clarity
# and grouped orig/reco arrays together logically
features = {key: ak.Array([]) for key in ["x", "y", "z", "e"]}  # original hit features
reco     = {key: ak.Array([]) for key in ["x", "y", "z", "e"]}  # embedder reconstructions

hit_labels = ak.Array([])
embeddings_orig, embeddings_aug = [], []
tokens = ak.Array([])

FEATURE_KEYS = ["x", "y", "z", "e"]  # CHANGE: named constant to avoid magic indices below

for n in range(num_files):

    file_dataset = loader(
        "ttbar_pu0_calo_hits",
        "train",
        start_idx=n * 100,
        stop_idx=(n + 1) * 100,
        train_fraction=1.0,
        E_min=E_min,
    )

    file_loader = DataLoader(
        file_dataset,
        batch_size=2,
        collate_fn=collater,
        num_workers=0,  # must be zero otherwise events are duplicated
        shuffle=False,  # CHANGE: fixed indentation (was misaligned)
    )

    for i, x_batch in tqdm(enumerate(file_loader)):

        if data_type == "hit":
            features_batch = x_batch["calo_hit_features"].to(embedder.device)
            mask_batch     = x_batch["mask"].to(embedder.device)

            # Augmentation must happen before standardization
            x_aug = augment_data(features_batch)
            x_aug = standardize_calo_hit_features_rphiz(x_aug)
            x_aug, vq_out_aug, z_embed_aug = embedder.forward(None, x_aug, mask_batch)

            x_orig = standardize_calo_hit_features_rphiz(features_batch)
            x_reco, vq_out, z_embed = embedder.forward(None, x_orig, mask_batch)  # first arg unused

        elif data_type == "patch":
            # CHANGE: removed debug print statements (print shape calls)
            e, e_reco, x_chunks, x_reco_chunks, vq_out = embedder.forward(x_batch, None, None)

        # Accumulate per-event arrays
        for row in range(x_reco.shape[0]):
            feat_event  = features_batch[row]
            reco_event  = x_reco[row].detach().cpu().numpy()
            mask_event  = mask_batch[row].int().detach().cpu().numpy()
            label_event = x_batch["hit_labels"][row]

            valid = mask_event == 1  # CHANGE: precompute boolean mask once per row

            # CHANGE: replaced 8 identical ak.concatenate lines with a loop over feature axes
            for col, key in enumerate(FEATURE_KEYS):
                features[key] = ak.concatenate([features[key], ak.Array([feat_event[:, col][valid]])], axis=0)
                reco[key]     = ak.concatenate([reco[key],     ak.Array([reco_event[:, col][valid]])], axis=0)

            hit_labels = ak.concatenate([hit_labels, ak.Array([label_event[valid]])], axis=0)

            embeddings_orig.append(z_embed[row][valid])
            embeddings_aug.append(z_embed_aug[row][valid])

            if vq_out is not None:
                tokens_event = vq_out["q"][row].detach().cpu().numpy().reshape(-1,)
                tokens = ak.concatenate([tokens, ak.Array([tokens_event[valid]])], axis=0)


In [21]:
# Shape: (num_events, num_hits, num_dimensions)
# CHANGE: removed dead commented-out code (torch.stack / .mean calls that were never used)
print(embeddings_orig.shape)
print(embeddings_aug.shape)


In [24]:
# CHANGE: moved pip install to its own cell (was already separate) — no code change needed
!pip install umap-learn


In [25]:
# CHANGE: removed duplicate 'import torch' and 'import numpy as np' / 'import matplotlib.pyplot as plt'
#         (already imported at the top of the notebook)
import umap


def plot_umap_side_by_side(embeddings_orig, embeddings_aug):
    """Project original and augmented embeddings to 2-D with UMAP and plot side-by-side."""
    # Move to CPU + convert to numpy if needed
    if isinstance(embeddings_orig, torch.Tensor):
        embeddings_orig = embeddings_orig.detach().cpu().numpy()
    if isinstance(embeddings_aug, torch.Tensor):
        embeddings_aug = embeddings_aug.detach().cpu().numpy()

    # Fit UMAP on combined data for a consistent projection space
    combined = np.concatenate([embeddings_orig, embeddings_aug], axis=0)
    reducer = umap.UMAP(n_components=2, random_state=42)
    embedding_2d = reducer.fit_transform(combined)

    # Split back into orig / aug halves
    n = embeddings_orig.shape[0]
    emb_orig_2d = embedding_2d[:n]
    emb_aug_2d  = embedding_2d[n:]

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # CHANGE: extracted shared axis-label/scatter kwargs to avoid repetition
    scatter_kw = dict(s=10)
    for ax, data, title in zip(axes,
                                [emb_orig_2d, emb_aug_2d],
                                ["Original Embeddings", "Augmented Embeddings"]):
        ax.scatter(data[:, 0], data[:, 1], **scatter_kw)
        ax.set_title(title)
        ax.set_xlabel("UMAP-1")
        ax.set_ylabel("UMAP-2")

    plt.tight_layout()
    plt.show()


In [26]:
plot_umap_side_by_side(embeddings_orig, embeddings_aug)


In [ ]:
print(len(features["x"]))


In [ ]:
# CHANGE: switched from integer-keyed dicts to string-keyed dicts to match the
#         FEATURE_KEYS naming introduced above; keeps all three dicts consistent
features_original = {
    0: features["x"],
    1: features["y"],
    2: features["z"],
    3: features["e"],
}

features_reconstructed = {
    0: reco["x"],
    1: reco["y"],
    2: reco["z"],
    3: reco["e"],
}

features_generated = {
    0: gen_x,
    1: gen_y,
    2: gen_z,
    3: gen_e,
}

print(len(features["x"]))
print(len(reco["x"]))
print(len(gen_x))


# Plot multiple events


In [ ]:
position_lim = 0.5
e_lim = -1, 1


## x, y, z, e (1d)

In [ ]:
# 1-D histograms of all hits across all events

nbins = 100

fig, ax = plt.subplots(1, 4, figsize=(20, 5))

# CHANGE: extracted repeated hist() calls into a helper to avoid 3×4 near-identical lines
def _plot_hist(axis, orig, reco, gen, bins):
    """Overlay orig / reco / gen histograms on *axis* with shared styling."""
    kw = dict(histtype="step", density=True)
    axis.hist(ak.flatten(orig), bins=bins, label="original",      **kw)
    axis.hist(ak.flatten(reco), bins=bins, label="reco (embedder)",**kw)
    axis.hist(ak.flatten(gen),  bins=bins, label="gen (gpt)",      **kw)

for i in range(3):
    bins = np.linspace(-position_lim, position_lim, nbins)
    _plot_hist(ax[i], features_original[i], features_reconstructed[i], features_generated[i], bins)
    ax[i].set_xlabel(f"$x_{i}$")

bins_e = np.linspace(e_lim[0], e_lim[1], nbins)
_plot_hist(ax[3], features_original[3], features_reconstructed[3], features_generated[3], bins_e)
ax[3].set_xlabel("e")
ax[3].set_yscale("log")
ax[3].legend()

ax[0].set_ylabel("Density")
plt.show()


In [ ]:
# Number-of-hits distribution per event

fig, ax = plt.subplots(1, 1, figsize=(4, 4))
bins = np.linspace(0, 8192, nbins)
kw   = dict(histtype="step", density=True)

# CHANGE: pulled repeated len-comprehension + hist calls into a small loop
for data, label in [
    (features_original,     "original"),
    (features_reconstructed,"reco (embedder)"),
    (features_generated,    "gen (gpt)"),
]:
    ax.hist([len(event) for event in data[0]], bins=bins, label=label, **kw)

ax.set_xlabel("Num. hits / event")
ax.set_ylabel("Density")
ax.legend()
plt.show()


# E resolution

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(4, 4))
ax.hist(
    (ak.flatten(features["e"]) - ak.flatten(reco["e"])) / ak.flatten(features["e"]),
    bins=nbins, histtype="step", label="original", density=True,
)
ax.set_xlabel("($E_{orig}$ - $E_{reco}$) / $E_{orig}$")
ax.set_ylabel("Density")
ax.set_yscale("log")
plt.show()


"""
fig, ax = plt.subplots(1, 1, figsize=(4, 4))

unique_labels = [np.unique(l.to_numpy()) for l in hit_labels]
hit_clusters_true, hit_clusters_reco = [], []
for event_i, hit_labels_event_i in enumerate(hit_labels):
    for unique_label_event_i in unique_labels[event_i]:
        mask = hit_labels_event_i == unique_label_event_i
   
        hit_clusters_true.append(np.sum(features["e"][event_i][mask]))
        hit_clusters_reco.append(np.sum(reco["e"][event_i][mask]))

ax.hist(
    (np.array(hit_clusters_true) - np.array(hit_clusters_reco)) / np.array(hit_clusters_true),
    bins=nbins, density=True, histtype="step", linewidth=2,
)
ax.set_xlabel("$E_{true} - E_{reco}$ / $E_{reco}$ per cluster")
ax.set_ylabel("Density")
ax.set_yscale("log")
# ax.legend(loc="upper right")
"""


# tokens

In [ ]:
# Token index distribution: original vs generated

fig, ax = plt.subplots(1, 1, figsize=(4, 4))
bins = np.arange(-0.5, 522.5, 1)
kw   = dict(histtype="step", density=True)
ax.hist(ak.flatten(tokens),     bins=bins, label="original",  **kw)
ax.hist(ak.flatten(gen_tokens), bins=bins, label="gen (gpt)", **kw)
ax.set_xlabel("Token index")
ax.set_ylabel("Density")
ax.legend()
plt.show()


# Plot single events

## x, y, z, e (1d)

In [ ]:
nbins = 100

for event_i in range(10):

    fig, ax = plt.subplots(1, 4, figsize=(20, 5))

    for i in range(3):
        bins = np.linspace(-position_lim, position_lim, nbins)
        # CHANGE: used _plot_hist helper defined above instead of 3 repeated hist() calls
        _plot_hist(ax[i],
                   features_original[i][event_i],
                   features_reconstructed[i][event_i],
                   features_generated[i][event_i],
                   bins)
        ax[i].set_xlabel(f"$x_{i}$")

    bins_e = np.linspace(e_lim[0], e_lim[1], nbins)
    _plot_hist(ax[3],
               features_original[3][event_i],
               features_reconstructed[3][event_i],
               features_generated[3][event_i],
               bins_e)
    ax[3].set_xlabel("e")
    ax[3].legend()
    ax[0].set_ylabel("Density")
    plt.show()

    fig, ax = plt.subplots(1, 1, figsize=(4, 4))
    ax.hist(tokens[event_i], bins=np.arange(-0.5, 522.5, 1), histtype="step", label="original")
    ax.legend()
    ax.set_xlabel("Token index")
    ax.set_ylabel("Density")
    plt.show()


## 2d positions

In [ ]:
detector_index = 9
detector_mask  = hit_labels == detector_index
print(detector_mask)
plot_lim = 0.2
s = 0.1

for i in range(5):

    fig, ax = plt.subplots(1, 3, figsize=(12, 4))

    for a in ax:
        a.set_box_aspect(1)  # true square subplot box
        a.set_xlim(-plot_lim, plot_lim)
        a.set_ylim(-plot_lim, plot_lim)

    # CHANGE: collapsed 3 nearly-identical scatter+label blocks into a loop
    for axis, data, title in zip(ax,
                                  [features_original, features_reconstructed, features_generated],
                                  ["original", "reco", "gen"]):
        axis.scatter(data[0][i], data[1][i], s=s)
        axis.set_xlabel("$x$")
        axis.set_ylabel("$y$")
        axis.set_title(title)

    plt.show()


In [ ]:
r_lim = 0., 0.3
z_lim = 0.6

for i in range(3):

    fig, ax = plt.subplots(1, 3, figsize=(12, 4))

    # CHANGE: removed commented-out set_box_aspect / set_xlim block
    # CHANGE: collapsed 3 near-identical scatter blocks into a loop
    for axis, data, title in zip(ax,
                                  [features_original, features_reconstructed, features_generated],
                                  ["original", "reco", "gen"]):
        tmp_r = np.sqrt(data[0][i] ** 2 + data[1][i] ** 2)
        axis.scatter(data[2][i], tmp_r, s=s)
        axis.set_xlabel("$z$")
        axis.set_ylabel("$r$")
        axis.set_xlim(-z_lim, z_lim)
        axis.set_ylim(r_lim)
        axis.set_title(title)

    plt.show()


# Apply augmentations and visualize latent space 